# Yelp — Baseline ANN (Plain MLP)
**Features**: 6 LOO numerical + OHE(state) + OHE(city) + multi-hot(categories) → dense
**Model**: Plain MLP — no embeddings
**Saves to**: `baseline_ANN/`


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Build feature matrix (dense for PyTorch)
STEP 4 — Dataset & DataLoader
STEP 5 — Model definition
STEP 6 — Train
STEP 7 — Evaluate
STEP 8 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from scipy.sparse          import csr_matrix, hstack
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")


In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\baseline_ANN"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET      = 'stars'
CAT_STATE   = 'state'
CAT_CITY    = 'city'
CAT_CATS    = 'categories'


---
## Step 3 — Build Feature Matrix (Dense)

Same OHE + multi-hot as classical models.
Converted to dense numpy array for PyTorch.

Note: the feature matrix is wide (~2,500+ columns) due to
OHE city (1,165) + multi-hot categories (1,281).
This is expected for the baseline — embeddings will compress this dramatically.


In [ ]:
# ── OHE: state (23 unique — low cardinality, clean OHE) ──
ohe_state = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
state_tr = ohe_state.fit_transform(df_train[[CAT_STATE]].fillna('Unknown'))
state_v  = ohe_state.transform(df_val[[CAT_STATE]].fillna('Unknown'))
state_te = ohe_state.transform(df_test[[CAT_STATE]].fillna('Unknown'))

# ── OHE: city (1,165 unique — medium cardinality) ──
ohe_city = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
city_tr = ohe_city.fit_transform(df_train[[CAT_CITY]].fillna('Unknown'))
city_v  = ohe_city.transform(df_val[[CAT_CITY]].fillna('Unknown'))
city_te = ohe_city.transform(df_test[[CAT_CITY]].fillna('Unknown'))

# ── Multi-hot: categories (1,281 unique tokens) ──
cats_tr = df_train[CAT_CATS].fillna('Unknown').str.get_dummies('|')
cat_cols = cats_tr.columns
cats_v  = df_val[CAT_CATS].fillna('Unknown').str.get_dummies('|').reindex(columns=cat_cols, fill_value=0)
cats_te = df_test[CAT_CATS].fillna('Unknown').str.get_dummies('|').reindex(columns=cat_cols, fill_value=0)

# ── Scale numerical features ──
scaler  = StandardScaler()
num_tr  = scaler.fit_transform(df_train[NUM_COLS])
num_v   = scaler.transform(df_val[NUM_COLS])
num_te  = scaler.transform(df_test[NUM_COLS])

# ── Stack all into sparse matrix ──
X_train = hstack([csr_matrix(num_tr), state_tr, city_tr, csr_matrix(cats_tr.values)])
X_val   = hstack([csr_matrix(num_v),  state_v,  city_v,  csr_matrix(cats_v.values)])
X_test  = hstack([csr_matrix(num_te), state_te, city_te, csr_matrix(cats_te.values)])

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

feat_names = (NUM_COLS
              + ohe_state.get_feature_names_out([CAT_STATE]).tolist()
              + ohe_city.get_feature_names_out([CAT_CITY]).tolist()
              + list(cat_cols))

print(f"Feature matrix — train: {X_train.shape}")
print(f"  {len(NUM_COLS)} numerical + {state_tr.shape[1]} state + "
      f"{city_tr.shape[1]} city + {cats_tr.shape[1]} categories")


In [ ]:
# Convert to dense for PyTorch
X_train_dense = X_train.toarray().astype(np.float32)
X_val_dense   = X_val.toarray().astype(np.float32)
X_test_dense  = X_test.toarray().astype(np.float32)
print(f"Dense matrix — train: {X_train_dense.shape}")


---
## Step 4 — Dataset & DataLoader

In [ ]:
class YelpDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE   = 512
train_loader = DataLoader(YelpDataset(X_train_dense, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(YelpDataset(X_val_dense,   y_val),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(YelpDataset(X_test_dense,  y_test),
                          batch_size=BATCH_SIZE, shuffle=False)

X_b, y_b = next(iter(train_loader))
print(f"Batch — X: {X_b.shape}  y: {y_b.shape}")


---
## Step 5 — Model Definition

Input dim is large (~2,500+) because OHE + multi-hot creates wide sparse features.
Architecture scales input down progressively: input → 256 → 128 → 64 → 1


In [ ]:
class PlainMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze(1)

model = PlainMLP(X_train_dense.shape[1]).to(DEVICE)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


---
## Step 6 — Train

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(X_b)
            loss  = criterion(preds, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(y_b.cpu().numpy())
    p = np.array(preds_all); l = np.array(labels_all)
    return (round(float(np.sqrt(mean_squared_error(l,p))),4),
            round(float(mean_absolute_error(l,p)),4))


In [ ]:
NUM_EPOCHS = 20
criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=0.001)
history    = {'train_rmse':[], 'val_rmse':[]}

t0 = time.perf_counter()
for epoch in range(NUM_EPOCHS):
    tr_rmse, _  = run_epoch(model, train_loader, criterion, optimizer)
    val_rmse, _ = run_epoch(model, val_loader,   criterion)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
              f"Train RMSE: {tr_rmse}  Val RMSE: {val_rmse}")
train_time = time.perf_counter() - t0
print(f"Training time: {train_time:.1f}s")


---
## Step 7 — Evaluate

In [ ]:
tr_rmse,  tr_mae  = run_epoch(model, train_loader, criterion)
val_rmse, val_mae = run_epoch(model, val_loader,   criterion)
te_rmse,  te_mae  = run_epoch(model, test_loader,  criterion)

print("=" * 50)
print("ANN — Baseline Results")
print("=" * 50)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.1f}s")
print(f"  Train-Test gap : {round(te_rmse - tr_rmse, 4)}")


---
## Step 8 — Save Results

In [ ]:
result = {
    'model'        : 'ANN (Baseline)',
    'train_rmse'   : tr_rmse, 'val_rmse'  : val_rmse, 'test_rmse'  : te_rmse,
    'train_mae'    : tr_mae,  'val_mae'   : val_mae,  'test_mae'   : te_mae,
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'ann_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'ann_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
